# Step 04 - Run Tensile Simulation

Run `main.py` with case-local inputs and generate stress-strain plot automatically.

In [ ]:
from pathlib import Path
import subprocess
import sys


def find_project_root(start: Path) -> Path:
    cur = start.resolve()
    for candidate in [cur, *cur.parents]:
        if (candidate / "main.py").exists() and (candidate / "al.gga.psp").exists():
            return candidate
    raise RuntimeError("Project root not found (expected main.py and al.gga.psp).")


ROOT = find_project_root(Path.cwd())
CASE_NAME = "nc_large_vac01"

CASE_DIR = (ROOT / "cases" / CASE_NAME).resolve()
INPUTS_DIR = (CASE_DIR / "inputs").resolve()

ECUT = 400
STEP = 0.005
RELAX_STEPS = 80
CYCLES = 120
FMAX = 0.08
PLOT_SUMMARY = True

for p in [INPUTS_DIR / "init.vasp", INPUTS_DIR / "bottom_idx.npy", INPUTS_DIR / "top_idx.npy", ROOT / "al.gga.psp"]:
    assert p.exists(), f"Missing: {p}"

print("ROOT:", ROOT)
print("CASE_DIR:", CASE_DIR)
print("INPUTS_DIR:", INPUTS_DIR)


In [ ]:
cmd = [
    sys.executable,
    str((ROOT / "main.py").resolve()),
    "--case", CASE_NAME,
    "--workdir", str(CASE_DIR),
    "--init", str((INPUTS_DIR / "init.vasp").resolve()),
    "--bottom-idx", str((INPUTS_DIR / "bottom_idx.npy").resolve()),
    "--top-idx", str((INPUTS_DIR / "top_idx.npy").resolve()),
    "--pp", str((ROOT / "al.gga.psp").resolve()),
    "--ecut", str(ECUT),
    "--step", str(STEP),
    "--relax-steps", str(RELAX_STEPS),
    "--cycles", str(CYCLES),
    "--fmax", str(FMAX),
]
if PLOT_SUMMARY:
    cmd.append("--plot-summary")

print("RUN CMD:\n", " ".join(cmd))
subprocess.run(cmd, check=True)


In [ ]:
from pathlib import Path
import glob
import csv

runs = sorted(glob.glob(str(CASE_DIR / "results" / f"{CASE_NAME}_*")))
assert runs, "No run folder found"
latest = Path(runs[-1])
print("Latest run:", latest)

summary = latest / "summary.csv"
plot = latest / "stress_strain.png"
print("Summary:", summary)
print("Plot   :", plot)

with open(summary, "r", encoding="utf-8") as f:
    rows = list(csv.reader(f))
print("Rows (including header):", len(rows))
print("Last row:", rows[-1])
